# R05 · Within-source-cell information

Test whether exact raw-derived subdivisions provide information missing from the original source cells, while keeping the returned ARM answer fixed.

**One question:** can raw-derived 2x2 subcells inside each original source cell improve prediction of the error of the
**same existing merged-histogram answer**? This is an information-sufficiency experiment. It does not change histogram
answers, introduce a new model family, impose a 5% accepted-risk target, or claim a compact/mergeable systems contribution.

## Run on Databricks

1. Pull this branch in your personal Git folder. Open this notebook and attach the same runtime used for notebook R03.
2. Run the configuration cell. Normally leave **`run_dir` blank**: it selects the latest completed **R03 objective-comparison**
   run (`error-objective-...`) under your personal `output_root`. If setting it explicitly, enter that complete result
   directory, **not** a `full-query-report-...` directory, notebook path or ZIP. The original R02 and R01 input runs must remain accessible.
3. Leave `raw_data_dir` blank if the matching raw files are in `<repo_root>/range_query`. Both `0.csv` and `0.csv.gz`
   are supported, similarly for other source IDs. Otherwise set the folder containing those files. Do not enter a single file.
   Each file must match its original `src_<id>` histogram. Files may have trailing timestamp columns; points are not weighted.
4. The dependency cell installs LightGBM only if missing, pinned to the selected source run's version. The remaining
   runtime package versions must match the source run; mismatches fail explicitly instead of silently changing the experiment.
5. For a small first run set `seeds=0`, then Run All. For the planned comparison use **`seeds=0,1,2`**, then Run All again.
   Each attempt creates a new `within-source-...` directory; previous outputs are preserved. Three seeds train **nine LightGBM models**.
6. Share **`within_cell_results.zip`** from the printed completed result directory. The ZIP contains predictions, compact
   reports, three figures and provenance. Raw points, histogram payloads and model weights are excluded and remain local.

`data_dir` and `histogram_dir` are optional relocation overrides for the original R01 files; hashes still must match.
`timing_queries=30` and `timing_repeats=5` affect only cost sampling. **All eligible test queries are used for model-quality metrics.**
No personal user path is hard-coded. If validation fails, share the printed failed-run ZIP; do not bypass parity checks.

## Fixed comparison

| Variant | Model inputs | Answer being assessed |
|---|---|---|
| `merged_only` | Existing 14 merged/query features | Original merged histogram |
| `full_source` | The same 14 plus the existing 7 source features | The same original merged histogram |
| `raw2_source` | The same 14 plus the **same 7 formulas**, evaluated on raw-derived 2x2 source subcells | The same original merged histogram |

Using the same seven formulas controls the feature dimension in the coarse-source versus raw-subcell comparison.
The subcell estimate is an **input feature only**, never the returned answer or replacement ground truth. No SA feature
is passed to these LightGBM models. Every arm uses raw-relative-error L1, the same non-objective parameters/tree cap,
the same outer cell split, the same grouped inner validation, and early stopping(100). Existing raw-L1 baselines must
reproduce saved notebook R03 test predictions. Calibration queries are held aside and unused; there is no tuned gate.

Raw SA is evaluated for ranking only. An additional, clearly labeled `sa_isotonic` baseline maps SA scores to numerical
relative error using **inner fitting data only**; it is not unmodified SA. A training-median constant checks whether the
learned numerical predictions provide useful information beyond a simple constant.

This test has already been inspected in earlier experiments. Results remain exploratory. A useful information signal
would justify testing compression/equal-budget finer histograms and then fresh held-out queries; it would not itself
establish a reliable, efficient SA replacement. The actual 2x2 payload can be **larger** than the source histograms.

In [ ]:
import ast
import base64
import gzip
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import platform
import shutil
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd

EPSILON = .25
VARIANTS = ('merged_only', 'full_source', 'raw2_source')
EARLY_STOPPING_ROUNDS = 100
INNER_FRACTION = .10
BOOTSTRAP_REPEATS = 2000


def notebook_defaults():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        root = ctx.notebookPath().get().split('/ml/notebooks/')[0]
        repo = root if root.startswith('/Workspace/') else '/Workspace' + root
        output = '/Workspace/Users/' + ctx.userName().get() + '/synopsislake-results'
    except (NameError, AttributeError):
        repo, output = str(Path.cwd()), str(Path(tempfile.gettempdir())/'synopsislake-results')
    return dict(repo_root=repo, output_root=output, run_dir='', raw_data_dir='',
                data_dir='', histogram_dir='', seeds='0,1,2', timing_queries='30', timing_repeats='5')


DEFAULTS = notebook_defaults()
try:
    for key, value in DEFAULTS.items():
        dbutils.widgets.text(key, value)
    get_config = dbutils.widgets.get
except NameError:
    def get_config(key):
        return os.environ.get(key.upper(), DEFAULTS[key])


def load_settings():
    cfg = {k: str(get_config(k)).strip() for k in DEFAULTS}
    cfg['seeds'] = [int(x) for x in cfg['seeds'].split(',')]
    for key in ('timing_queries', 'timing_repeats'):
        cfg[key] = int(cfg[key])
        if cfg[key] < 1:
            raise ValueError(key + ' must be positive')
    if not cfg['seeds'] or len(set(cfg['seeds'])) != len(cfg['seeds']):
        raise ValueError('Use distinct seeds present in the completed objective-comparison run')
    return cfg


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False, default=str))
    temporary.replace(path)


def deterministic_ids(ids, count, salt):
    return sorted(sorted(set(map(int, ids)),
        key=lambda q: hashlib.sha256(f'{salt}:{q}'.encode()).digest())[:count])


def resolve_run(cfg):
    if cfg['run_dir']:
        source = Path(cfg['run_dir']).expanduser().resolve()
    else:
        eligible = []
        for path in Path(cfg['output_root']).expanduser().glob('*/run.json'):
            info = json.loads(path.read_text())
            if (info.get('status') == 'complete' and info.get('study') == 'error-objective-comparison'
                    and set(cfg['seeds']) <= set(info.get('config', {}).get('seeds', []))):
                eligible.append(path)
        if not eligible:
            raise FileNotFoundError('Set run_dir to the completed notebook 13 error-objective-... directory; not full-query-report, 12 or a ZIP.')
        source = max(eligible, key=lambda p: p.stat().st_mtime_ns).parent
    prior = json.loads((source/'run.json').read_text())
    if (prior.get('status') != 'complete' or prior.get('study') != 'error-objective-comparison'
            or not set(cfg['seeds']) <= set(prior['config']['seeds'])):
        raise ValueError('run_dir must be a complete notebook 13 error-objective-comparison run containing the requested seeds')
    for name in ('outer_splits.csv', 'inner_validation_assignments.csv', 'feature_definitions.json',
                 'test_predictions.csv', 'models.csv'):
        if not (source/name).is_file():
            raise FileNotFoundError('Missing source artifact: ' + str(source/name))
    print('Using notebook 13 objective comparison:', source, flush=True)
    return source, prior

In [ ]:
# INSTALL_DEPENDENCIES
# Run before importing training libraries. Preserve the source run's tested version.
import importlib.util
import sys
_config_for_install = load_settings()
_source_for_install, _prior_for_install = resolve_run(_config_for_install)
if importlib.util.find_spec('lightgbm') is None:
    _required_lgb = _prior_for_install['package_versions']['lightgbm']
    get_ipython().run_line_magic('pip', 'install lightgbm==' + _required_lgb)
else:
    print('LightGBM is installed; the experiment will verify its version against the source run.')

## Reuse the existing pipeline

Only hash-pinned definition cells/functions are loaded. No training, widgets or reports from notebooks R02/13 execute.
AST matching tolerates outputs, presentation metadata and missing Databricks cell IDs, but changed definitions fail.
Workspace export uses the signed-in runtime identity when Git notebooks are not available as ordinary files.
The inherited preparation still builds auxiliary 12 representations; its cost is recorded separately from raw-subcell construction.

In [ ]:
REUSE_SPECS = {'group-validation-04': None, 'group-validation-06': ['make_inner_assignments', 'prepare_experiment'], 'group-validation-16': ['objective_predictions', 'fit_error_objective']}
REUSE_HASHES = {'group-validation-04': '1cb91647979d06a1d0025c1ab7505c5331294eede355abe470ae9718a5d2de9f', 'group-validation-06': 'a1c7f5456aced7d6f1ebb5a4e68875f06d105f6ad826673b71e18d59ae2283d8', 'group-validation-16': '89abb65953383614ee714bda677bb683e954a27bd1a9402dc56b9be23ae888c3'}


def read_document(path):
    try:
        raw = Path(path).read_bytes()
        doc = json.loads(raw)
        method = 'workspace filesystem'
    except (OSError, ValueError, UnicodeError):
        if not str(path).startswith('/Workspace/'):
            raise
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service.workspace import ExportFormat
        client = WorkspaceClient()
        wp = str(path)[len('/Workspace'):]
        last = None
        for candidate in (wp, wp.removesuffix('.ipynb')):
            try:
                result = client.workspace.export(path=candidate, format=ExportFormat.JUPYTER)
                raw = base64.b64decode(result.content)
                doc = json.loads(raw)
                method = 'Workspace API using runtime identity'
                break
            except Exception as error:
                last = error
        else:
            raise ValueError('Cannot read notebook 13. Check repo_root and workspace access: '+str(last)) from last
    return doc, dict(path=str(path), read_method=method, document_sha256=hashlib.sha256(raw).hexdigest())


def load_notebook13(repo):
    path = Path(repo).resolve()/'ml/notebooks/15_group_validation.ipynb'
    doc, provenance = read_document(path)
    env = dict(globals())
    env['__name__'] = 'verified_notebook13_definitions'
    env.update(INNER_FRACTION=.10, EARLY_STOPPING_ROUNDS=100,
               VARIANTS=('merged_only', 'compact16', 'full_source'), POLICIES=('query', 'cell'))
    for label, names in REUSE_SPECS.items():
        matches = []
        for cell in doc['cells']:
            if cell['cell_type'] != 'code':
                continue
            try:
                tree = ast.parse(''.join(cell['source']))
            except SyntaxError:
                continue
            if names is not None:
                nodes = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name in names]
                if {n.name for n in nodes} != set(names):
                    continue
                tree = ast.Module(body=nodes, type_ignores=[])
            elif any(not isinstance(n, (ast.Import, ast.ImportFrom, ast.Assign, ast.FunctionDef)) for n in tree.body):
                continue
            digest = hashlib.sha256(ast.dump(tree, include_attributes=False).encode()).hexdigest()
            if digest == REUSE_HASHES[label]:
                matches.append(tree)
        if len(matches) != 1:
            raise ValueError('Notebook 13 definitions changed/missing: '+label+'. Pull matching notebooks 12, 13 and 15.')
        exec(compile(matches[0], str(path)+':'+label, 'exec'), env)
    provenance['definition_ast_sha256'] = REUSE_HASHES
    return env, provenance


def prepare_fixed_data(cfg, source, prior, destination, state):
    env, provenance = load_notebook13(cfg['repo_root'])
    source12 = Path(prior['source12_run'])
    if sha(source12/'run.json') != prior['source12_manifest_sha256']:
        raise ValueError('Notebook 12 manifest changed since the selected 13 run')
    prepare_cfg = dict(cfg, run_dir=str(source12))
    data, namespace, inherited, outer, inner, _ = env['prepare_experiment'](prepare_cfg, destination, state)
    if state['package_versions'] != prior['package_versions']:
        raise ValueError('Use the same runtime package versions as the source objective comparison')
    old_definitions = json.loads((source/'feature_definitions.json').read_text())
    if data['definitions'] != old_definitions:
        raise ValueError('Feature definitions changed since the source objective comparison')
    original_outer = pd.read_csv(source/'outer_splits.csv')
    original_outer = original_outer[original_outer.seed.isin(cfg['seeds'])]
    original_inner = pd.read_csv(source/'inner_validation_assignments.csv')
    original_inner = original_inner[(original_inner.validation_policy == 'cell') & original_inner.seed.isin(cfg['seeds'])]
    for left, right, keys, sort in (
            (outer, original_outer, ['seed', 'qId', 'cell', 'split'], ['seed', 'qId']),
            (inner[inner.validation_policy == 'cell'], original_inner, ['seed', 'qId', 'cell', 'role'], ['seed', 'qId'])):
        pd.testing.assert_frame_equal(left[keys].sort_values(sort).reset_index(drop=True),
            right[keys].sort_values(sort).reset_index(drop=True), check_dtype=False)
    original_outer.to_csv(destination/'outer_splits.csv', index=False)
    original_inner.to_csv(destination/'inner_validation_assignments.csv', index=False)
    # Retain the old feature map only as an implementation dependency; this experiment does not train compact16.
    data['frames'] = {v: data['frames'][v] for v in ('merged_only', 'full_source')}
    manifest = json.loads((destination/'preparation/input_manifest.json').read_text())
    source_names = sorted(Path(k).stem for k in manifest if k.startswith('run/src_') and k.endswith('.txt'))
    data['source_ids'] = [int(n.removeprefix('src_')) for n in source_names]
    if len(data['source_ids']) != len(data['source_contexts']):
        raise ValueError('Source file identities do not match prepared contexts')
    data['definitions'] = {v: data['definitions'][v] for v in ('merged_only', 'full_source')}
    params = dict(data['gate_helpers'].PARAMS, n_estimators=inherited['max_trees'], n_jobs=inherited['num_threads'])
    if params != prior['base_model_parameters']:
        raise ValueError('Non-objective LightGBM parameters differ from the source run')
    state.update(reuse_notebook13=provenance, source13_run=str(source), source13_manifest_sha256=sha(source/'run.json'),
        source13_artifact_sha256={name:sha(source/name) for name in
            ('outer_splits.csv','inner_validation_assignments.csv','test_predictions.csv','feature_definitions.json','models.csv')},
        base_model_parameters=params, learning_objective='relative_l1', variants=list(VARIANTS))
    return data, namespace, env, inherited, original_outer, original_inner, params

## Build metadata without query labels

For each original source cell, split both axes at the midpoint and count the raw points in its four subcells.
`src_i` corresponds to raw file `i.csv` or `i.csv.gz`; overlapping source extents do not duplicate point ownership.
Before training, every reconstructed coarse count must match the original source histogram, and subcell mass must be conserved.
Wrong data, malformed coordinates, out-of-extent points and source-count mismatches stop the run; no rescaling is used.

The source histogram uses Java-style uniform half-open binning; a midpoint belongs to the right/upper child.
Independent exact query scans are inclusive on all four sides, matching notebook T01. Thirty-two audit queries are chosen
by a fixed hash of qId, without looking at error or prediction. This small input audit does not reduce the full test cohort.
The metadata builder accepts no query rectangles, truth, split or labels. Construction and raw-audit times are separate.
Raw point counts are serialized losslessly as uint32 (uint64 if needed), with float64 boundaries, in uncompressed payloads.

In [ ]:
def refine_boundaries(boundaries):
    b = np.asarray(boundaries, float)
    if b.ndim != 1 or len(b) < 2 or not np.isfinite(b).all() or (np.diff(b) <= 0).any():
        raise ValueError('Invalid source boundaries')
    refined = np.empty(2*len(b)-1)
    refined[::2] = b
    refined[1::2] = b[:-1] + np.diff(b)/2
    if (np.diff(refined) <= 0).any():
        raise ValueError('Source cells are too narrow to subdivide in float64')
    return refined


def iter_raw_points(path, chunk_size=100000):
    # Same first-two-TAB-fields, one-row-one-point semantics as notebook 09 / Java PointScanCounter.
    # Extra columns (e.g., timestamps) are ignored; duplicate rows remain duplicate points.
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as stream:
        for part in pd.read_csv(stream, sep='\t', header=None, usecols=[0, 1], dtype=float,
                                chunksize=chunk_size, float_precision='round_trip', skip_blank_lines=False):
            points = part.to_numpy(float)
            if points.shape[1] != 2 or not np.isfinite(points).all():
                raise ValueError('Invalid/nonfinite raw coordinates in '+str(path))
            yield points


def coarse_indices(points, context):
    indices = []
    for axis, b in enumerate((context['bx'], context['by'])):
        values = points[:, axis]
        if ((values < b[0]) | (values >= b[-1])).any():
            raise ValueError('Raw point outside half-open source extent; wrong raw file or source export')
        # Java uses this uniform arithmetic, not a newly inferred grid or spatial source assignment.
        width = (b[-1]-b[0])/(len(b)-1)
        if not np.allclose(np.diff(b), width, rtol=1e-8, atol=1e-12):
            raise ValueError('Expected the original uniform source histogram')
        ix = ((values-b[0])/width).astype(np.int64)
        if ((ix < 0) | (ix >= len(b)-1)).any():
            raise ValueError('Uniform source index outside grid; inspect boundary rounding')
        indices.append(ix)
    return tuple(indices)


def resolve_raw_paths(cfg, data):
    directory = Path(cfg['raw_data_dir']).expanduser() if cfg['raw_data_dir'] else Path(cfg['repo_root'])/'range_query'
    paths = []
    for i in data['source_ids']:
        candidates = [directory/f'{i}.csv', directory/f'{i}.csv.gz']
        path = next((p for p in candidates if p.is_file()), None)
        if path is None:
            raise FileNotFoundError('Missing raw source '+str(i)+'. Set raw_data_dir to the folder containing 0.csv, 1.csv, ... (or .csv.gz). Tried '+str(directory))
        paths.append(path.resolve())
    return paths


def save_raw_payload(path, hist):
    rate, bx, by, counts = hist
    if not np.equal(counts, np.floor(counts)).all() or (counts < 0).any():
        raise ValueError('Raw point counts must be nonnegative integers')
    dtype = np.uint32 if counts.max(initial=0) <= np.iinfo(np.uint32).max else np.uint64
    # Uncompressed archive: actual bytes, no data-dependent compression advantage in the storage comparison.
    np.savez(path, version=np.array([1], np.uint32), rate=np.array([rate]),
             bx=np.asarray(bx, np.float64), by=np.asarray(by, np.float64), counts=counts.astype(dtype))


def load_raw_payload(path):
    with np.load(path, allow_pickle=False) as archive:
        if set(archive.files) != {'version','rate','bx','by','counts'} or archive['version'].tolist() != [1]:
            raise ValueError('Unsupported raw subcell payload')
        result = (float(archive['rate'][0]), archive['bx'].copy(), archive['by'].copy(), archive['counts'].astype(float))
    if result[3].shape != (len(result[1])-1, len(result[2])-1) or not np.isfinite(result[3]).all():
        raise ValueError('Invalid raw subcell payload dimensions')
    return result


def build_raw_subcells(raw_paths, source_contexts, destination, namespace, state):
    if len(raw_paths) != len(source_contexts):
        raise ValueError('Each source histogram requires its corresponding raw file')
    folder = destination/'raw_payloads'; folder.mkdir()
    rows, inputs, contexts, paths = [], [], [], []
    for path, source in zip(raw_paths, source_contexts):
        # Preserve actual file IDs even if lexical source ordering includes src_10 before src_2.
        i = Path(path).name.split('.')[0]
        before = sha(path)
        bx, by = np.asarray(source['bx']), np.asarray(source['by'])
        nx, ny = source['C'].shape
        coarse = np.zeros((nx, ny), np.int64)
        fine = np.zeros((2*nx, 2*ny), np.int64)
        tick = time.perf_counter(); count = 0
        for points in iter_raw_points(path):
            ix, iy = coarse_indices(points, source)
            hx = points[:, 0] >= (bx[ix] + (bx[ix+1]-bx[ix])/2)
            hy = points[:, 1] >= (by[iy] + (by[iy+1]-by[iy])/2)
            coarse += np.bincount(ix*ny+iy, minlength=nx*ny).reshape(nx, ny)
            fine += np.bincount((2*ix+hx)*(2*ny)+(2*iy+hy), minlength=4*nx*ny).reshape(2*nx, 2*ny)
            count += len(points)
        elapsed = time.perf_counter()-tick
        gap = float(np.max(np.abs(coarse-source['C'])))
        mass_match = bool(np.array_equal(fine.reshape(nx, 2, ny, 2).sum(axis=(1,3)), coarse))
        rows.append(dict(source_id=i, raw_rows=count, source_total=float(source['C'].sum()),
            max_coarse_count_gap=gap, subcell_mass_conserved=mass_match, build_scan_seconds=elapsed))
        pd.DataFrame(rows).to_csv(destination/'raw_source_parity.csv', index=False)
        if before != sha(path):
            raise ValueError('Raw source changed during construction: '+str(path))
        if gap > 1e-6 or not mass_match or count != float(source['C'].sum()):
            raise ValueError('Raw/source coarse counts do not match for source '+str(i)+'. Inspect raw_source_parity.csv; no rescaling is performed.')
        payload = folder/f'src_{i}-raw2.npz'
        save_raw_payload(payload, (source['rate'], refine_boundaries(bx), refine_boundaries(by), fine))
        loaded = load_raw_payload(payload)
        contexts.append(namespace['histogram_context'](loaded)); paths.append(payload)
        inputs.append(dict(source_id=i, raw_path=str(path), raw_sha256=before, raw_file_bytes=Path(path).stat().st_size,
            payload_path=str(payload), payload_sha256=sha(payload), payload_bytes=payload.stat().st_size))
        save_json(destination/'raw_input_manifest.json', inputs)
        print('Raw source', i, 'rows', count, 'coarse counts verified; subcell bytes', payload.stat().st_size, flush=True)
    state['timings']['raw_metadata_build_scan_seconds'] = sum(r['build_scan_seconds'] for r in rows)
    return contexts, paths


def raw_truth_parity(raw_paths, queries, qids, destination):
    # Independent direct scans are an audit only; the metadata builder never receives query rectangles or truth.
    counts = np.zeros(len(qids), np.int64)
    manifest_path = destination/'raw_input_manifest.json'
    expected = {r['raw_path']:r['raw_sha256'] for r in json.loads(manifest_path.read_text())} if manifest_path.is_file() else {}
    tick = time.perf_counter()
    for path in raw_paths:
        before = sha(path)
        if expected and before != expected.get(str(path)):
            raise ValueError('Raw source changed after metadata construction: '+str(path))
        for points in iter_raw_points(path):
            for j, qid in enumerate(qids):
                q = queries[qid, :4]
                counts[j] += np.count_nonzero((points[:,0] >= q[0]) & (points[:,0] <= q[1]) &
                                             (points[:,1] >= q[2]) & (points[:,1] <= q[3]))
        if before != sha(path):
            raise ValueError('Raw source changed during truth audit')
    table = pd.DataFrame(dict(qId=qids, saved_truth=queries[qids,4], raw_exact_count=counts))
    table['matches'] = np.isclose(table.saved_truth, table.raw_exact_count, rtol=0, atol=1e-6)
    table.to_csv(destination/'raw_truth_parity.csv', index=False)
    if not table.matches.all():
        raise ValueError('Raw counts differ from query truth. Inspect raw_truth_parity.csv; do not interpret model results.')
    return time.perf_counter()-tick

## Fit and evaluate every test query

The original merged estimate, saved truth and `abs(estimate-truth)/truth` are identical in all arms and verified before fitting.
The inherited cohort excludes true counts below 10, including zero. Predictions are floored at zero and never upper-clipped.
Inner grouped validation selects the tree count using L1; neither test nor outer calibration labels select models or gates.
Full-test MAE/p90/bias are in **percentage points**. AUC measures discrimination of error >25%, not numerical prediction accuracy.
Risk-coverage curves retain complete ties and are descriptive ranking diagnostics, not deployment thresholds.

In [ ]:
def attach_raw_features(data, namespace, contexts, destination, state):
    original = list(data['definitions']['full_source'][14:])
    if len(original) != 7 or not all(n.startswith('src_') for n in original):
        raise ValueError('Expected the existing seven source-feature formulas')
    renamed = {name:'raw2_'+name for name in original}
    qids = sorted(set(data['frames']['merged_only'].qId.astype(int)))
    tick = time.perf_counter()
    # No truth, error, split, model score or query-specific exact answer is passed to the feature provider.
    features = pd.DataFrame([namespace['source_query'](data['queries'][qid,:4], contexts,
                              data['merged_helpers']) for qid in qids], index=qids).rename(columns=renamed)
    table = data['frames']['merged_only'].join(features, on='qId', validate='many_to_one')
    data['frames']['raw2_source'] = table
    data['definitions']['raw2_source'] = list(data['definitions']['merged_only']) + list(renamed.values())
    state['timings']['offline_raw_subcell_feature_seconds'] = time.perf_counter()-tick
    for variant in VARIANTS:
        part = data['frames'][variant]
        reference = data['frames']['merged_only']
        keys = ['qId','histogram_key','truth','estimate','relErr']
        pd.testing.assert_frame_equal(part[keys].reset_index(drop=True), reference[keys].reset_index(drop=True))
        if not np.isfinite(part[data['definitions'][variant]].to_numpy(float)).all():
            raise ValueError('Nonfinite query features: '+variant)
    save_json(destination/'feature_definitions.json', data['definitions'])
    state['feature_counts'] = {v:len(data['definitions'][v]) for v in VARIANTS}
    return renamed


def prediction_metrics(errors, predicted, scores):
    errors, scores = np.asarray(errors,float), np.asarray(scores,float)
    result = dict(test_count=len(errors), bad_fraction=float(np.mean(errors > EPSILON)),
                  auc_bad=float(roc_auc_score(errors > EPSILON, scores)) if len(np.unique(errors > EPSILON)) == 2 else np.nan)
    if predicted is not None:
        residual = np.asarray(predicted,float)-errors
        result.update(mae_pp=float(100*np.mean(np.abs(residual))), p90_ae_pp=float(100*np.quantile(np.abs(residual),.9)),
            bias_pp=float(100*np.mean(residual)), rmse_pp=float(100*np.sqrt(np.mean(residual**2))))
    else:
        result.update(mae_pp=np.nan, p90_ae_pp=np.nan, bias_pp=np.nan, rmse_pp=np.nan)
    return result


def risk_coverage_rows(errors, scores):
    order = np.argsort(scores, kind='stable')
    s, errors = np.asarray(scores)[order], np.asarray(errors)[order]
    ends = np.flatnonzero(np.r_[s[1:] != s[:-1], True])
    bad = np.cumsum(errors > EPSILON)
    # All complete-tie endpoints, not a test-selected deployment threshold.
    return [dict(accepted_count=int(k+1), violation_count=int(bad[k]),
                 coverage=float((k+1)/len(s)), violation=float(bad[k]/(k+1))) for k in ends]


def fit_and_evaluate(cfg, source, data, namespace13, params, outer, inner, destination, state):
    from sklearn.isotonic import IsotonicRegression
    saved = pd.read_csv(source/'test_predictions.csv')
    prediction_rows, metrics, audits, models, curves = [], [], [], [], []
    for seed in cfg['seeds']:
        split = outer[outer.seed == seed]
        assignment = inner[inner.seed == seed]
        fit_ids = set(assignment.loc[assignment.role == 'fit', 'qId'])
        validation_ids = set(assignment.loc[assignment.role == 'validation', 'qId'])
        training_ids = set(split.loc[split.split == 'train', 'qId'])
        if fit_ids & validation_ids or fit_ids | validation_ids != training_ids:
            raise ValueError('Invalid frozen training partition')
        test_ids = set(split.loc[split.split == 'test', 'qId'])
        if training_ids & test_ids or (training_ids | test_ids) & set(split.loc[split.split == 'calibration','qId']):
            raise ValueError('Outer roles overlap')
        for variant in VARIANTS:
            frame, names = data['frames'][variant], data['definitions'][variant]
            training = frame[frame.qId.isin(training_ids)]
            tick = time.perf_counter()
            model, actual, validation_metric, nfit, nval = namespace13['fit_error_objective'](
                training, names, fit_ids, validation_ids, seed, 'relative_l1', params)
            elapsed = time.perf_counter()-tick
            folder = destination/'models'/f'seed_{seed}'/variant; folder.mkdir(parents=True)
            model.booster_.save_model(str(folder/'model.txt'))
            save_json(folder/'schema.json', dict(features=names, params=actual, seed=seed, variant=variant,
                target='abs(fixed_merged_estimate-truth)/truth', prediction_transform='max(raw_prediction,0)',
                outer_split='cell', inner_validation='cell', refit_on_outer_train=False, min_exact=10))
            models.append(dict(seed=seed, variant=variant, fit_seconds=elapsed,
                tree_count=model.booster_.num_trees(), best_iteration=int(model.best_iteration_),
                model_bytes=(folder/'model.txt').stat().st_size, model_sha256=sha(folder/'model.txt'),
                fit_rows=nfit, validation_rows=nval, **validation_metric))
            for base in data['base_names']:
                test = frame[(frame.histogram_key == base) & frame.qId.isin(test_ids)].sort_values('qId')
                native = model.predict(test[names], num_threads=params['n_jobs'])
                predicted = np.maximum(native, 0.)
                if not np.isfinite(predicted).all():
                    raise ValueError('Nonfinite model prediction')
                prefix = dict(seed=seed, variant=variant, base_rate=data['rates'][base])
                metrics.append(dict(**prefix, **prediction_metrics(test.relErr, predicted, predicted)))
                prediction_rows.append(test[['qId','histogram_key','truth','estimate','relErr']].assign(
                    **prefix, raw_prediction=native, predicted_error=predicted, score=predicted))
                curves.extend(dict(**prefix,**r) for r in risk_coverage_rows(test.relErr, predicted))
                if variant != 'raw2_source':
                    reference = saved[(saved.seed == seed) & (saved.variant == variant) &
                        (saved.validation_policy == 'cell') & (saved.learning_objective == 'relative_l1') &
                        np.isclose(saved.base_rate, data['rates'][base], rtol=0, atol=1e-12)].sort_values('qId')
                    matches = (len(reference) == len(test) and np.array_equal(reference.qId, test.qId)
                        and np.allclose(reference.predicted_error, predicted, rtol=1e-9, atol=1e-10)
                        and np.allclose(reference.relErr, test.relErr, rtol=1e-12, atol=1e-12))
                    audits.append(dict(**prefix, scores_and_targets_match=bool(matches)))
                    pd.DataFrame(audits).to_csv(destination/'baseline_reproduction.csv', index=False)
                    if not matches:
                        raise ValueError('Existing raw-L1 baseline does not reproduce notebook 13; inspect baseline_reproduction.csv')
            print('seed', seed, variant, 'features', len(names), 'trees', model.booster_.num_trees(),
                  'validation MAE pp', round(validation_metric['validation_mae_pp'],3), flush=True)
        # Predeclared numerical SA baseline: fit a monotone score-to-error map using inner FIT only.
        # Original SA is retained separately for ranking; no raw SA numerical MAE is invented.
        frame = data['frames']['merged_only']
        fit = frame[frame.qId.isin(fit_ids)]
        if not np.isfinite(frame.logged_sa).all():
            raise ValueError('Missing/nonfinite original SA logger scores')
        iso = IsotonicRegression(increasing=True, out_of_bounds='clip', y_min=0.)
        iso.fit(fit.logged_sa.to_numpy(), fit.relErr.to_numpy())
        folder = destination/'models'/f'seed_{seed}'; folder.mkdir(exist_ok=True)
        save_json(folder/'sa_isotonic.json', dict(x=iso.X_thresholds_.tolist(), y=iso.y_thresholds_.tolist(),
            fitted_role='inner fit only', increasing=True, out_of_bounds='clip'))
        constant = float(fit.relErr.median())
        for base in data['base_names']:
            test = frame[(frame.histogram_key == base) & frame.qId.isin(test_ids)].sort_values('qId')
            for variant, predicted, score in (
                    ('sa_raw', None, test.logged_sa.to_numpy()),
                    ('sa_isotonic', iso.predict(test.logged_sa.to_numpy()), iso.predict(test.logged_sa.to_numpy())),
                    ('constant_train_median', np.full(len(test), constant), np.full(len(test), constant))):
                prefix = dict(seed=seed, variant=variant, base_rate=data['rates'][base])
                metrics.append(dict(**prefix, **prediction_metrics(test.relErr,predicted,score)))
                prediction_rows.append(test[['qId','histogram_key','truth','estimate','relErr']].assign(
                    **prefix, raw_prediction=np.nan, predicted_error=np.nan if predicted is None else predicted, score=score))
                curves.extend(dict(**prefix,**r) for r in risk_coverage_rows(test.relErr,score))
    metrics = pd.DataFrame(metrics); predictions = pd.concat(prediction_rows, ignore_index=True)
    if predictions.duplicated(['seed','variant','base_rate','qId']).any():
        raise ValueError('Duplicate prediction keys')
    metrics.to_csv(destination/'all_query_metrics.csv', index=False)
    predictions.to_csv(destination/'test_predictions.csv', index=False)
    pd.DataFrame(models).to_csv(destination/'models.csv', index=False)
    pd.DataFrame(curves).to_csv(destination/'risk_coverage.csv', index=False)
    metrics.groupby(['variant','base_rate'], as_index=False).agg(
        seeds=('seed','nunique'), mean_mae_pp=('mae_pp','mean'), mean_p90_ae_pp=('p90_ae_pp','mean'),
        mean_bias_pp=('bias_pp','mean'), mean_auc_bad=('auc_bad','mean')).to_csv(destination/'summary.csv', index=False)
    return predictions, metrics

## Report practical effect and cost separately

Paired MAE differences compare the same qId/seed/physical histogram. Negative differences favor raw2_source.
For uncertainty, repeated seed appearances are averaged per query, then existing source-cell groups are resampled.
These exploratory intervals condition on the fitted models. Seeds/merge rates are not independent query samples, and
inferred source-cell groups do not establish strict geographic generalization.

Serialized bytes, model weights, construction, training, payload/context initialization and model loading are separate.
Resident single-query timing includes the original histogram estimate, query features, array preparation, inference and
fixed zero-floor. All models receive identical hash-selected timing queries for each seed/rate; live scores must match batch scores.
Contexts and models are resident, with one warm-up per query. Filesystem caches are uncontrolled. There is no raw read
inside the timed query loop, no remote-retrieval/SA timing, no exact fallback and no E2E speedup claim.

In [ ]:
def paired_effects(predictions, data, destination):
    rows = []
    candidate = predictions[predictions.variant == 'raw2_source']
    for reference_name in ('merged_only','full_source','sa_isotonic','constant_train_median'):
        reference = predictions[predictions.variant == reference_name]
        pair = candidate.merge(reference, on=['seed','base_rate','qId'], suffixes=('_new','_ref'), validate='one_to_one')
        if len(pair) != len(candidate) or not np.allclose(pair.relErr_new, pair.relErr_ref, rtol=0, atol=1e-12):
            raise ValueError('Paired comparison has different queries or targets')
        pair['ae_delta_pp'] = 100*(np.abs(pair.predicted_error_new-pair.relErr_new)-
                                   np.abs(pair.predicted_error_ref-pair.relErr_ref))
        # Average repeated appearances BEFORE resampling; seeds/rates are not independent observations.
        for rate, part in pair.groupby('base_rate'):
            by_query = part.groupby('qId').ae_delta_pp.mean().to_frame()
            by_query['cell'] = data['cells'].loc[by_query.index].astype(str).to_numpy()
            clusters = by_query.groupby('cell').ae_delta_pp.agg(['sum','count'])
            rng = np.random.default_rng(15001)
            boot = []
            if len(clusters) >= 2:
                for _ in range(BOOTSTRAP_REPEATS):
                    ix = rng.integers(0,len(clusters),len(clusters))
                    boot.append(float(clusters['sum'].to_numpy()[ix].sum()/clusters['count'].to_numpy()[ix].sum()))
            rows.append(dict(reference=reference_name, base_rate=rate, unique_queries=len(by_query),
                unique_cell_groups=len(clusters), mean_ae_delta_pp=float(by_query.ae_delta_pp.mean()),
                ci95_low_pp=float(np.quantile(boot,.025)) if boot else np.nan,
                ci95_high_pp=float(np.quantile(boot,.975)) if boot else np.nan,
                bootstrap_repeats=len(boot), negative_delta_favors_raw2=True,
                uncertainty_scope='exploratory cell-cluster bootstrap conditional on fitted models; repeated seeds averaged per query'))
    pd.DataFrame(rows).to_csv(destination/'paired_effects.csv', index=False)


def record_payloads(data, raw_paths, destination):
    rows = []
    for base in data['base_names']:
        for variant in VARIANTS:
            extra = data['source_paths'] if variant == 'full_source' else raw_paths if variant == 'raw2_source' else []
            rows.append(dict(variant=variant, base_rate=data['rates'][base], base_histogram_bytes=data['paths'][base].stat().st_size,
                auxiliary_bytes=sum(p.stat().st_size for p in extra),
                total_synopsis_bytes=data['paths'][base].stat().st_size+sum(p.stat().st_size for p in extra),
                model_weights_included=False, python_memory_included=False,
                auxiliary_representation='raw-derived 2x2 source subhistograms' if variant == 'raw2_source' else variant))
    pd.DataFrame(rows).to_csv(destination/'payloads.csv', index=False)


def measure_resident_costs(cfg, data, namespace, raw_contexts, raw_paths, params, outer, destination):
    records, loads = [], []
    # Explicit local file/context initialization; filesystem/OS caches are uncontrolled, so these are not cold-storage reads.
    for base in data['base_names']:
        for variant in VARIANTS:
            tick = time.perf_counter()
            namespace['histogram_context'](namespace['load_hist_payload'](data['paths'][base]))
            if variant == 'full_source':
                for path in data['source_paths']:
                    namespace['histogram_context'](namespace['load_hist_payload'](path))
            elif variant == 'raw2_source':
                for path in raw_paths:
                    namespace['histogram_context'](load_raw_payload(path))
            loads.append(dict(variant=variant, base_rate=data['rates'][base], kind='payload_and_context_init',
                              seconds=time.perf_counter()-tick, cache='uncontrolled local filesystem cache'))
    saved = pd.read_csv(destination/'test_predictions.csv')
    for seed in cfg['seeds']:
        test_ids = outer.loc[(outer.seed == seed) & (outer.split == 'test'),'qId']
        selected = deterministic_ids(test_ids, cfg['timing_queries'], f'15-timing:{seed}')
        for variant in VARIANTS:
            tick = time.perf_counter()
            model = lgb.Booster(model_file=str(destination/'models'/f'seed_{seed}'/variant/'model.txt'))
            loads.append(dict(variant=variant, seed=seed, kind='model_load', seconds=time.perf_counter()-tick,
                              cache='uncontrolled local filesystem cache'))
            names = data['definitions'][variant]
            sources = data['source_contexts'] if variant == 'full_source' else raw_contexts if variant == 'raw2_source' else None
            for base in data['base_names']:
                ctx = data['contexts'][base]
                expected = saved[(saved.seed == seed) & (saved.variant == variant) &
                    np.isclose(saved.base_rate, data['rates'][base], rtol=0, atol=1e-12)].set_index('qId')
                for qid in selected:
                    q = data['queries'][qid,:4]
                    # One warm-up per query/model. Runtime features never receive raw paths, truth or labels.
                    warm = namespace['extract_query'](q,ctx,data['merged_helpers'],sources=sources)
                    if variant == 'raw2_source':
                        warm.update({'raw2_'+n:v for n,v in list(warm.items()) if n.startswith('src_')})
                    model.predict(np.array([[warm[n] for n in names]]),num_threads=params['n_jobs'])
                    for repeat in range(cfg['timing_repeats']):
                        start = time.perf_counter_ns()
                        f = namespace['extract_query'](q,ctx,data['merged_helpers'],sources=sources)
                        if variant == 'raw2_source':
                            f.update({'raw2_'+n:v for n,v in list(f.items()) if n.startswith('src_')})
                        array = np.array([[f[n] for n in names]],float)
                        middle = time.perf_counter_ns()
                        score = max(float(model.predict(array,num_threads=params['n_jobs'])[0]),0.)
                        stop = time.perf_counter_ns()
                        if not np.isclose(score, expected.loc[qid,'predicted_error'], rtol=1e-9, atol=1e-10):
                            raise ValueError('Live prediction differs from saved batch prediction')
                        records.append(dict(seed=seed, variant=variant, base_rate=data['rates'][base], qId=qid, repeat=repeat,
                            feature_ms=(middle-start)/1e6, inference_ms=(stop-middle)/1e6, total_ms=(stop-start)/1e6,
                            cache='resident contexts and model; one warm-up per query; no raw reads'))
    pd.DataFrame(loads).to_csv(destination/'initialization.csv', index=False)
    timing = pd.DataFrame(records); timing.to_csv(destination/'resident_timings.csv', index=False)
    rows=[]
    for key,part in timing.groupby(['seed','variant','base_rate']):
        row=dict(zip(('seed','variant','base_rate'),key), timed_unique_queries=part.qId.nunique(),repeats=cfg['timing_repeats'])
        for metric in ('feature_ms','inference_ms','total_ms'):
            row['mean_'+metric]=float(part[metric].mean());row['p95_'+metric]=float(part[metric].quantile(.95))
        rows.append(row)
    pd.DataFrame(rows).to_csv(destination/'resident_timing_summary.csv', index=False)

## Save results

Three figures show full-query prediction error, ranking quality over selected fractions, and payload/resident-query cost.
`analysis_en.md` is generated in the result directory only. Nothing is written into the Git checkout other than this notebook.
New directories are never overwritten. Failed runs preserve their available audit CSVs and manifest in the same ZIP format.

In [ ]:
def write_report(destination):
    import matplotlib.pyplot as plt
    metrics=pd.read_csv(destination/'all_query_metrics.csv')
    payloads=pd.read_csv(destination/'payloads.csv')
    timing=pd.read_csv(destination/'resident_timing_summary.csv')
    curves=pd.read_csv(destination/'risk_coverage.csv')
    variants=list(VARIANTS)+['sa_isotonic','constant_train_median']
    # Full-query averages are descriptive averages over seeds; no independent-trial error bars are drawn.
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for ax,metric,label in zip(axes,('mae_pp','p90_ae_pp'),('Mean absolute prediction error (pp)','90th percentile prediction error (pp)')):
        values=metrics.groupby('variant')[metric].mean().reindex(variants)
        ax.bar(range(len(variants)),values);ax.set_xticks(range(len(variants)),variants,rotation=25,ha='right')
        ax.set_ylabel(label)
    fig.suptitle('All test queries; fixed histogram answers');fig.tight_layout();fig.savefig(destination/'prediction_error.png',dpi=150);plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,5))
    for variant in list(VARIANTS)+['sa_raw']:
        fractions=np.linspace(.01,1,100);lines=[]
        for _,part in curves[curves.variant==variant].groupby(['seed','base_rate']):
            part=part.sort_values('coverage')
            ix=np.searchsorted(part.coverage.to_numpy(),fractions,side='left').clip(max=len(part)-1)
            lines.append(part.violation.to_numpy()[ix])
        ax.plot(fractions,np.mean(lines,axis=0),label=variant)
    ax.set(xlabel='Nominal selected fraction (complete ties retained)',ylabel='Violation fraction among selected answers',
           title='Ranking diagnostic; actual coverage can exceed nominal coverage')
    ax.legend();fig.tight_layout();fig.savefig(destination/'risk_coverage.png',dpi=150);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    a=payloads.groupby('variant').total_synopsis_bytes.mean().reindex(VARIANTS)/1024
    b=timing.groupby('variant').mean_total_ms.mean().reindex(VARIANTS)
    axes[0].bar(a.index,a);axes[0].set_ylabel('Serialized synopsis payload (KiB; model excluded)')
    axes[1].bar(b.index,b);axes[1].set_ylabel('Mean resident feature + inference time (ms)')
    for ax in axes:ax.tick_params(axis='x',rotation=20)
    fig.suptitle('Information prototype costs; no E2E or equal-budget claim');fig.tight_layout();fig.savefig(destination/'costs.png',dpi=150);plt.close(fig)
    text='''# Within-source information: exploratory result

This experiment holds the original merged-histogram answers and true errors fixed. It compares identical raw-relative-error L1 LightGBM settings, with the exact outer and inner grouped splits inherited from notebook 13. It does not impose a 5% acceptance-risk target or choose a test threshold.

## Full-query prediction

'''+metrics.groupby('variant')[['mae_pp','p90_ae_pp','bias_pp','auc_bad']].mean().round(4).to_csv()+'''
`mae_pp` measures the difference between predicted and actual relative error in percentage points, not classification accuracy. Raw SA scores have no numerical error MAE. `sa_isotonic` is an additional monotone score-to-error mapping fitted only on the inner fitting data. It is not the unmodified paper baseline. The constant uses the inner-fitting error median.

## Interpretation and limits

- Compare raw2_source with both merged_only and full_source. Improvement shows value of this raw-derived spatial representation, not a general proof that raw data are necessary or that a neural model cannot help.
- Read paired_effects.csv: negative MAE differences favor raw2. Repeated appearances across seeds are averaged per query before a cell-cluster bootstrap. Intervals condition on these fitted models; source-cell groups are not strict geographic separation and the existing test has already been inspected.
- Raw 2x2 source counts are themselves finer histograms. This prototype is not automatically a compact, mergeable or novel SA replacement. Actual payload/model sizes and resident timings must be considered. A same-budget finer-answer comparison and fresh held-out queries remain necessary for a final systems claim.
- The serialized subcells are built without query labels, using all matching underlying data as a synopsis normally would. Raw coarse-cell counts must reproduce every source histogram, and fixed hash-selected queries are checked with independent inclusive raw scans. The sampled audit does not replace full-query model evaluation.
- Source binning is half-open; query truth scans are closed on all four sides. Uniform subcell overlap estimates are heuristic and are not certified bounds for points on boundaries.
- Query-time measurement uses resident payloads and models, includes feature extraction/histogram estimation and inference, and performs no raw reads. File/context and model initialization, metadata construction, raw parity scans and model training are recorded separately. No live SA timing, remote retrieval, exact fallback, E2E speedup or deployment guarantee is measured.
- True counts below 10, including zero, are excluded by the inherited cohort. Relative error is abs(estimate-truth)/truth; there is no zero-count accuracy claim.

If raw2 provides no useful full-query improvement, do not continue tuning this representation on the same test. If it helps, freeze the candidate and evaluate its quality/space/time tradeoff against a same-budget finer histogram, then validate on fresh queries.
'''
    (destination/'analysis_en.md').write_text(text)


def export_results(destination):
    with ZipFile(destination/'within_cell_results.zip','w',ZIP_DEFLATED) as archive:
        # Share diagnostics and predictions only; never raw points, models or histogram payloads.
        for path in sorted(destination.iterdir()):
            if path.is_file() and path.suffix in {'.csv','.json','.md','.png'}:
                archive.write(path,path.name)


def run_experiment(cfg):
    global lgb, uniform_filter, roc_auc_score
    import lightgbm as lgb
    from scipy.ndimage import uniform_filter
    from sklearn.metrics import roc_auc_score
    destination=Path(cfg['output_root']).expanduser().resolve()/('within-source-'+uuid4().hex[:12])
    destination.mkdir(parents=True,exist_ok=False)
    state=dict(status='running',study='within-source-information',config=cfg,timings={},
        started_utc=datetime.now(timezone.utc).isoformat(),python=platform.python_version(),epsilon=EPSILON,
        protocol=dict(subdivision=2,learning_objective='relative_l1',early_stopping_rounds=100,
            acceptance_risk_target=None,test_selected_threshold=False,
            raw_semantics='TAB first two fields; one row one point; duplicate rows retained; trailing fields ignored',
            construction='query-independent per-source 2x2 subcells from matching raw files',
            same_histogram_answers=True, exploratory=True))
    save_json(destination/'run.json',state);started=time.perf_counter()
    print('Writing new run:',destination,flush=True)
    try:
        source,prior=resolve_run(cfg)
        data,namespace,env,inherited,outer,inner,params=prepare_fixed_data(cfg,source,prior,destination,state)
        save_json(destination/'run.json',state)
        raw_paths=resolve_raw_paths(cfg,data)
        tick=time.perf_counter()
        raw_contexts,payloads=build_raw_subcells(raw_paths,data['source_contexts'],destination,namespace,state)
        state['timings']['raw_construction_including_hashes_and_payload_setup_seconds']=time.perf_counter()-tick
        eligible=sorted(set(data['frames']['merged_only'].qId.astype(int)))
        audit_ids=deterministic_ids(eligible,32,'15-raw-truth-parity-v1')
        state['timings']['independent_raw_truth_audit_seconds']=raw_truth_parity(raw_paths,data['queries'],audit_ids,destination)
        attach_raw_features(data,namespace,raw_contexts,destination,state)
        record_payloads(data,payloads,destination)
        save_json(destination/'run.json',state)
        tick=time.perf_counter()
        predictions,metrics=fit_and_evaluate(cfg,source,data,env,params,outer,inner,destination,state)
        state['timings']['model_fitting_and_evaluation_seconds']=time.perf_counter()-tick
        paired_effects(predictions,data,destination)
        measure_resident_costs(cfg,data,namespace,raw_contexts,payloads,params,outer,destination)
        write_report(destination)
        state.update(status='complete',completed_utc=datetime.now(timezone.utc).isoformat(),
            total_seconds=time.perf_counter()-started,full_test_prediction_rows=len(predictions),
            raw_truth_audit_queries=len(audit_ids),raw_truth_audit_selection='fixed hash of qId, independent of outcomes')
        save_json(destination/'run.json',state);export_results(destination)
        print('Completed:',destination,flush=True)
        print('Share:',destination/'within_cell_results.zip',flush=True)
        return destination
    except Exception as error:
        state.update(status='failed',error_type=type(error).__name__,error=str(error),total_seconds=time.perf_counter()-started)
        save_json(destination/'run.json',state);export_results(destination)
        print('Failed run diagnostics:',destination/'within_cell_results.zip',flush=True)
        raise

## Execute and share

Run this cell after the definitions above. The output directory is printed immediately when selected and again on completion.
Send **`within_cell_results.zip`**; no separate file selection is needed. Review `raw_source_parity.csv`, `raw_truth_parity.csv`
and `baseline_reproduction.csv` before interpreting quality. Main results are `all_query_metrics.csv`, `summary.csv`,
`paired_effects.csv`, `payloads.csv`, `models.csv`, `resident_timing_summary.csv` and the three PNG figures.

In [ ]:
# RUN_WITHIN_SOURCE_EXPERIMENT
config = load_settings()
result_dir = run_experiment(config)
print(pd.read_csv(result_dir / 'summary.csv').to_string(index=False))